I want to shift the analysis from simply counting eBird checklists to checking whether those checklists actually contain reported bird observations. That could give me a much stronger comparison for my Chicago maps.

I want to compare:
1. Where people submitted birdwatching checklists.
2. Where those checklists contain at least one recorded bird observation.

# Where Chicago's Birds Get Counted

### Working narrative

I started this project interested in the uneven distribution of citizen-science reporting across Chicago. Using eBird records from 2020 through 2024, I mapped 218,781 checklist submissions across the city's 77 community areas. The geographic differences were enormous: Uptown alone accounted for 39,441 submissions, while some neighborhoods had almost no recorded activity.

Initially, I wanted to understand why certain parts of Chicago received so much more birdwatching attention than others. Popular birding destinations clearly play a role. Montrose Point, for example, accounts for 32,320 checklist submissions. But looking more closely at who was generating the data revealed another explanation: a relatively small number of accounts contribute a disproportionate share of Chicago's reports.

The most striking example was one account responsible for 12,405 checklists across 2023 and 2024. In Edgewater, this single account generated 71% of all checklist submissions during the five-year study period. It also accounted for approximately 45% of Rogers Park's reports and 32% of West Ridge's. Those figures raise an important question about what the geographical patterns actually represent.

I investigated this observer's reporting behavior and found that most of their 2024 checklists were extremely short, with a median duration of three minutes. Many were submitted from individual street corners and parks around Edgewater. Rather than necessarily reflecting separate, lengthy birdwatching outings, these reports appear consistent with a much more frequent and geographically specific recording style.

However, these were not simply empty submissions. Of the 10,915 checklists this account submitted in 2024, 10,832 had matching bird observation records. This distinction matters: unusually frequent reporting does not automatically mean low-quality or uninformative reporting.

My project is now moving beyond checklist counts to investigate the birds recorded within those checklists. I'm interested in how the number of birds and species reported varies across locations and months, particularly during spring and fall migration. Ultimately, I want to distinguish patterns in the birds being observed from patterns created by the people observing them.

**The central question is no longer just where Chicago's birds are being counted, but how the people doing the counting shape what we can learn from the data.**

In [ ]:
from src.bird_colors import BIRD_COLORS, MAP_PALETTES, OBSERVER_COLORS
from src.build_datasets import load_data, build_datasets

tables = build_datasets(load_data())

community_map = tables["community_map"]
hotspots = tables["hotspot_summary"]
monthly = tables["monthly_activity"]

In [1]:
import pandas as pd
import altair as alt
import geopandas as gpd

checklists = pd.read_csv(
    "../data/processed/chicago_checklists_2020_2024.csv"
)

observations_file = "../data/raw/ebird/cook_ebird_observations_2020_2025.txt"

print("Chicago checklists:", len(checklists))
print("Unique checklist IDs:",
      checklists["SAMPLING EVENT IDENTIFIER"].nunique())

print(checklists.columns.tolist())

Chicago checklists: 218781
Unique checklist IDs: 218781
['LAST EDITED DATE', 'COUNTRY', 'COUNTRY CODE', 'STATE', 'STATE CODE', 'COUNTY', 'COUNTY CODE', 'IBA CODE', 'BCR CODE', 'USFWS CODE', 'ATLAS BLOCK', 'LOCALITY', 'LOCALITY ID', 'LOCALITY TYPE', 'LATITUDE', 'LONGITUDE', 'OBSERVATION DATE', 'TIME OBSERVATIONS STARTED', 'OBSERVER ID', 'OBSERVER ORCID ID', 'SAMPLING EVENT IDENTIFIER', 'OBSERVATION TYPE', 'PROTOCOL NAME', 'PROTOCOL CODE', 'PROJECT NAMES', 'PROJECT IDENTIFIERS', 'DURATION MINUTES', 'EFFORT DISTANCE KM', 'EFFORT AREA HA', 'NUMBER OBSERVERS', 'ALL SPECIES REPORTED', 'GROUP IDENTIFIER', 'CHECKLIST COMMENTS', 'Unnamed: 33', 'community']


In [10]:
community_areas = gpd.read_file(
    "../data/processed/community_areas.geojson"
)

species_map = community_areas.merge(
    community_species,
    on="community",
    how="left"
)

species_map["species_count"] = species_map["species_count"].fillna(0)

In [2]:
# collect the identifiers for every Chicago checklist
checklist_ids = set(checklists["SAMPLING EVENT IDENTIFIER"])

# track which checklists have at least one bird observation
observed_ids = set()

for chunk in pd.read_csv(
    observations_file,
    sep="\t",
    usecols=[
        "SAMPLING EVENT IDENTIFIER",
        "COMMON NAME",
        "OBSERVATION COUNT"
    ],
    chunksize=200_000
):
    # only observations matching my Chicago checklists
    matched = chunk[
        chunk["SAMPLING EVENT IDENTIFIER"].isin(checklist_ids)
    ].copy()

    # xclude explicit zero counts and any no-birds placeholder
    counts = pd.to_numeric(
        matched["OBSERVATION COUNT"],
        errors="coerce"
    )

    matched = matched[
        matched["COMMON NAME"].notna()
        & ~matched["COMMON NAME"].eq("No birds observed")
        & ~counts.eq(0)
    ]

    observed_ids.update(
        matched["SAMPLING EVENT IDENTIFIER"].unique()
    )

print("Checklists with bird observations:", len(observed_ids))

Checklists with bird observations: 216690


In [11]:
cols = [
    "SAMPLING EVENT IDENTIFIER",
    "CATEGORY",
    "COMMON NAME",
    "SCIENTIFIC NAME",
    "OBSERVATION COUNT"
]

parts = []

for chunk in pd.read_csv(
    observations_file,
    sep="\t",
    usecols=cols,
    chunksize=200_000
):
    matched = chunk[
        chunk["SAMPLING EVENT IDENTIFIER"].isin(checklist_ids)
    ]

    if not matched.empty:
        parts.append(matched)

bird_records = pd.concat(parts, ignore_index=True)

# attach checklist information to each observation
birds = bird_records.merge(
    checklists[
        ["SAMPLING EVENT IDENTIFIER", "OBSERVER ID",
         "OBSERVATION DATE", "community",
         "DURATION MINUTES", "OBSERVATION TYPE"]
    ],
    on="SAMPLING EVENT IDENTIFIER",
    how="left",
    validate="many_to_one"
)

birds.to_csv(
    "../data/processed/chicago_bird_observations_2020_2024.csv.gz",
    index=False,
    compression="gzip"
)

print("Bird observation records:", len(birds))
display(birds.head(50))

Bird observation records: 2840621


,CATEGORY,COMMON NAME,SCIENTIFIC NAME,OBSERVATION COUNT,SAMPLING EVENT IDENTIFIER,OBSERVER ID,OBSERVATION DATE,community,DURATION MINUTES,OBSERVATION TYPE
0,species,American Black Duck,Anas rubripes,1,S63207282,obsr17234,2020-01-09,LINCOLN PARK,10.0,Traveling
1,species,American Black Duck,Anas rubripes,2,S63117311,obsr92042,2020-01-03,SOUTH DEERING,5.0,Traveling
2,species,American Black Duck,Anas rubripes,2,S62867949,obsr819571,2020-01-01,SOUTH DEERING,11.0,Traveling
3,species,American Black Duck,Anas rubripes,1,S63047636,obsr90711,2020-01-05,WEST PULLMAN,6.0,Traveling
4,species,American Black Duck,Anas rubripes,1,S62920060,obsr662292,2020-01-02,LINCOLN PARK,40.0,Traveling
5,species,American Black Duck,Anas rubripes,2,S63390056,obsr28258,2020-01-14,EAST SIDE,39.0,Stationary
6,species,American Black Duck,Anas rubripes,1,S62860590,obsr399437,2020-01-01,SOUTH DEERING,18.0,Traveling
7,species,American Black Duck,Anas rubripes,2,S63825208,obsr607938,2020-01-26,EAST SIDE,37.0,Traveling
8,species,American Black Duck,Anas rubripes,4,S63682558,obsr142929,2020-01-22,WEST PULLMAN,45.0,Stationary
9,species,American Black Duck,Anas rubripes,2,S63610411,obsr1336264,2020-01-20,NORTH CENTER,15.0,Stationary


I want to know how often a certain species is reported AND how many individual birds are reported when it is encountered. I wonder if maybe many people are seeing canadian geese but just 1 at a time, where as a cardinal could be seen fewer times but in bulk.

checklists = Number of distinct checklists that reported that species.
birds_reported = Total numeric counts of that species across those checklists

I have 218,781 checklists that contain 0-many birds. 89,783/218,781 of those checklists included a canada goose

In [4]:
species = birds[birds["CATEGORY"] == "species"].copy()

species["bird_count"] = pd.to_numeric(
    species["OBSERVATION COUNT"],
    errors="coerce"
)

species_rank = (
    species.groupby("COMMON NAME")
    .agg(
        checklists=("SAMPLING EVENT IDENTIFIER", "nunique"),
        birds_reported=("bird_count", "sum")
    )
    .reset_index()
)

print("Most frequently reported species:")
display(species_rank.nlargest(20, "checklists"))

print("Largest reported bird counts:")
display(species_rank.nlargest(20, "birds_reported"))

Most frequently reported species:


,COMMON NAME,checklists,birds_reported
15,American Robin,111897,952374.0
146,House Sparrow,111789,1428185.0
187,Northern Cardinal,93704,262370.0
65,Canada Goose,89783,3401722.0
241,Ring-billed Gull,85262,2450987.0
174,Mallard,80024,921118.0
110,European Starling,73185,1243583.0
37,Black-capped Chickadee,63308,192381.0
238,Red-winged Blackbird,59008,754604.0
9,American Goldfinch,58062,312629.0


Largest reported bird counts:


,COMMON NAME,checklists,birds_reported
65,Canada Goose,89783,3401722.0
241,Ring-billed Gull,85262,2450987.0
146,House Sparrow,111789,1428185.0
110,European Starling,73185,1243583.0
228,Red-breasted Merganser,23073,974406.0
15,American Robin,111897,952374.0
174,Mallard,80024,921118.0
94,Double-crested Cormorant,36467,847895.0
238,Red-winged Blackbird,59008,754604.0
257,Sandhill Crane,1884,520394.0


The Sandhill Crane result is especially interesting. It appears on relatively few checklists but has an enormous reported count. That is consistent with sightings of large migrating flocks and look into checklist records and months. Northern cardinals are frequently reported but not in flocks. 

In [5]:
checklist_birds = (
    species.groupby("SAMPLING EVENT IDENTIFIER")
    .agg(
        birds_reported=("bird_count", "sum"),
        species_reported=("SCIENTIFIC NAME", "nunique")
    )
    .reset_index()
)

display(checklist_birds.head(15))

,SAMPLING EVENT IDENTIFIER,birds_reported,species_reported
0,S100005173,18.0,2
1,S100005193,18.0,2
2,S100009116,1.0,1
3,S100009748,4.0,4
4,S100011558,1.0,1
5,S100012231,27.0,3
6,S100012756,22.0,6
7,S100013766,7.0,3
8,S100014442,206.0,11
9,S100015410,102.0,12


In [6]:
total_birds = checklist_birds["birds_reported"].sum()

print(f"Total Chicago checklists: {len(checklists):,}")
print(f"Total birds reported: {total_birds:,.0f}")
print(f"Checklists with species records: {len(checklist_birds):,}")

Total Chicago checklists: 218,781
Total birds reported: 22,957,739
Checklists with species records: 214,323


one person submits a checklist recording:
- 10 American Robins
- 5 Canada Geese
- 2 Northern Cardinals
That equals one checklist, three species, and 17 birds reported.

In [7]:
print("Distinct bird species:", species["SCIENTIFIC NAME"].nunique())

Distinct bird species: 335


In [8]:
community_species = (
    species.groupby("community")["SCIENTIFIC NAME"]
    .nunique()
    .reset_index(name="species_count")
)

display(
    community_species.sort_values(
        "species_count", ascending=False
    )
)

,community,species_count
66,UPTOWN,300
62,SOUTH CHICAGO,263
48,NEAR SOUTH SIDE,261
65,SOUTH SHORE,256
38,LINCOLN PARK,253
...,...,...
14,CHATHAM,21
67,WASHINGTON HEIGHTS,20
6,AVALON PARK,17
25,FULLER PARK,14


In [9]:
species_chart = alt.Chart(species_map).mark_geoshape(
    stroke="white",
    strokeWidth=0.5
).encode(
    color=alt.Color(
        "species_count:Q",
        title="Distinct bird species",
        scale=alt.Scale(
            range=["#EDF5FB", "#B6D7EA", "#65A6C9", "#2872A2", "#123B64"]
        )
    ),
    tooltip=[
        alt.Tooltip("community:N", title="Community area"),
        alt.Tooltip("species_count:Q", title="Species reported", format=",")
    ]
).project(
    type="mercator"
).properties(
    title="Diversity of Bird Species Reported Across Chicago, 2020–2024",
    width=500,
    height=600
)

species_chart

NameError: name 'species_map' is not defined

In [ ]:
species_ids = set(species["SAMPLING EVENT IDENTIFIER"])

other_ids = observed_ids - species_ids

print("Checklists with bird records but no species records:", len(other_ids))

display(
    birds[birds["SAMPLING EVENT IDENTIFIER"].isin(other_ids)]
    ["CATEGORY"].value_counts()
)

Checklists with bird records but no species records: 2367


CATEGORY
domestic    1806
spuh         344
issf         267
slash         65
form          22
hybrid        14
Name: count, dtype: int64

In [ ]:
facts = {
    "period": "2020–2024",
    "chicago_checklists": 218781,
    "birds_reported": 22957739,
    "distinct_species": 335,
    "ebird_accounts": 10373,
    "community_areas": 77,
    "most_reported_species": "American Robin",
    "robin_checklists": 111897,
    "largest_numeric_species_count": "Canada Goose",
    "goose_birds_reported": 3401722,
    "most_reported_community": "UPTOWN",
    "uptown_checklists": 39441,
    "least_reported_community": "BURNSIDE",
    "burnside_checklists": 1,
    "top_hotspot": "Montrose Point Bird Sanctuary",
    "montrose_checklists": 32320,
    "busiest_checklist_month": "May",
    "top_20_pct_account_share": 89.3,
    "checklists_with_bird_records": 216690,
    "checklists_with_species_records": 214323,
    "edgewater_top_account_share": 71.1
}
facts

{'period': '2020–2024',
 'chicago_checklists': 218781,
 'birds_reported': 22957739,
 'distinct_species': 335,
 'ebird_accounts': 10373,
 'community_areas': 77,
 'most_reported_species': 'American Robin',
 'robin_checklists': 111897,
 'largest_numeric_species_count': 'Canada Goose',
 'goose_birds_reported': 3401722,
 'most_reported_community': 'UPTOWN',
 'uptown_checklists': 39441,
 'least_reported_community': 'BURNSIDE',
 'burnside_checklists': 1,
 'top_hotspot': 'Montrose Point Bird Sanctuary',
 'montrose_checklists': 32320,
 'busiest_checklist_month': 'May',
 'top_20_pct_account_share': 89.3,
 'checklists_with_bird_records': 216690,
 'checklists_with_species_records': 214323,
 'edgewater_top_account_share': 71.1}

In [ ]:
# I identify checklists with at least one recorded bird observation
checklists["has_bird_record"] = (
    checklists["SAMPLING EVENT IDENTIFIER"].isin(observed_ids)
)

print("=== ALL CHICAGO CHECKLISTS ===")

display(
    checklists["has_bird_record"]
    .value_counts()
    .rename(index={
        True: "With a bird observation",
        False: "Without a matching bird observation"
    })
)

print("\nPercentage with bird observations:")
print(round(checklists["has_bird_record"].mean() * 100, 2))

=== ALL CHICAGO CHECKLISTS ===


has_bird_record
With a bird observation                216690
Without a matching bird observation      2091
Name: count, dtype: int64


Percentage with bird observations:
99.04


In [ ]:
# compare the most prolific account with the full Chicago dataset
observer = checklists[
    checklists["OBSERVER ID"] == "obsr4476574"
].copy()

observer["year"] = pd.to_datetime(
    observer["OBSERVATION DATE"]
).dt.year

print("=== OBSERVER 4476574 ===")

display(
    observer.groupby(["year", "has_bird_record"])
    .size()
    .unstack(fill_value=0)
)

print("\n2024 checklists with recorded birds:")
display(
    observer[observer["year"] == 2024]
    ["has_bird_record"].value_counts()
)

=== OBSERVER 4476574 ===


has_bird_record,False,True
year,,
2023,125,1365
2024,83,10832



2024 checklists with recorded birds:


has_bird_record
True     10832
False       83
Name: count, dtype: int64

In [ ]:
# check the community-area column name
print([
    col for col in checklists.columns
    if "comm" in col.lower()
])

['CHECKLIST COMMENTS', 'community']


In [ ]:
#compare total checklists with those containing bird observations
community_observations = (
    checklists.groupby("community")
    .agg(
        total_checklists=("SAMPLING EVENT IDENTIFIER", "nunique"),
        checklists_with_birds=("has_bird_record", "sum"),
        observers=("OBSERVER ID", "nunique")
    )
    .reset_index()
)

community_observations["without_bird_record"] = (
    community_observations["total_checklists"]
    - community_observations["checklists_with_birds"]
)

community_observations["pct_with_birds"] = (
    community_observations["checklists_with_birds"]
    / community_observations["total_checklists"]
    * 100
)

display(
    community_observations.sort_values(
        "without_bird_record", ascending=False
    ).head(15)
)

,community,total_checklists,checklists_with_birds,observers,without_bird_record,pct_with_birds
66,UPTOWN,39441,39255,3799,186,99.528410
21,EDGEWATER,9282,9110,336,172,98.146951
75,WEST TOWN,7093,6936,956,157,97.786550
51,NORTH CENTER,10524,10384,1002,140,98.669707
34,IRVING PARK,8622,8487,657,135,98.434238
60,ROGERS PARK,6634,6532,369,102,98.462466
37,LAKE VIEW,7945,7853,1265,92,98.842039
39,LINCOLN SQUARE,9929,9843,1028,86,99.133850
38,LINCOLN PARK,12191,12117,1938,74,99.392995
0,ALBANY PARK,5721,5651,369,70,98.776438


In [ ]:
# compare reporting activity with recorded bird observations
community_rank = (
    checklists.groupby("community")
    .agg(
        total_checklists=("SAMPLING EVENT IDENTIFIER", "nunique"),
        with_birds=("has_bird_record", "sum"),
        observers=("OBSERVER ID", "nunique")
    )
    .reset_index()
)

community_rank["without_birds"] = (
    community_rank["total_checklists"] - community_rank["with_birds"]
)

community_rank["pct_with_birds"] = (
    100 * community_rank["with_birds"] / community_rank["total_checklists"]
)

community_rank["pct_without_birds"] = (
    100 - community_rank["pct_with_birds"]
)

print("Most checklists submitted:")
display(community_rank.nlargest(15, "total_checklists"))

print("Most checklists without matching bird records:")
display(community_rank.nlargest(15, "without_birds"))

print("Highest percentage without matching bird records (100+ checklists):")
display(
    community_rank[community_rank["total_checklists"] >= 100]
    .nlargest(15, "pct_without_birds")
)

Most checklists submitted:


,community,total_checklists,with_birds,observers,without_birds,pct_with_birds,pct_without_birds
66,UPTOWN,39441,39255,3799,186,99.528410,0.471590
38,LINCOLN PARK,12191,12117,1938,74,99.392995,0.607005
51,NORTH CENTER,10524,10384,1002,140,98.669707,1.330293
39,LINCOLN SQUARE,9929,9843,1028,86,99.133850,0.866150
53,NORTH PARK,9469,9426,1383,43,99.545887,0.454113
21,EDGEWATER,9282,9110,336,172,98.146951,1.853049
34,IRVING PARK,8622,8487,657,135,98.434238,1.565762
37,LAKE VIEW,7945,7853,1265,92,98.842039,1.157961
48,NEAR SOUTH SIDE,7159,7129,1366,30,99.580947,0.419053
75,WEST TOWN,7093,6936,956,157,97.786550,2.213450


Most checklists without matching bird records:


,community,total_checklists,with_birds,observers,without_birds,pct_with_birds,pct_without_birds
66,UPTOWN,39441,39255,3799,186,99.528410,0.471590
21,EDGEWATER,9282,9110,336,172,98.146951,1.853049
75,WEST TOWN,7093,6936,956,157,97.786550,2.213450
51,NORTH CENTER,10524,10384,1002,140,98.669707,1.330293
34,IRVING PARK,8622,8487,657,135,98.434238,1.565762
60,ROGERS PARK,6634,6532,369,102,98.462466,1.537534
37,LAKE VIEW,7945,7853,1265,92,98.842039,1.157961
39,LINCOLN SQUARE,9929,9843,1028,86,99.133850,0.866150
38,LINCOLN PARK,12191,12117,1938,74,99.392995,0.607005
0,ALBANY PARK,5721,5651,369,70,98.776438,1.223562


Highest percentage without matching bird records (100+ checklists):


,community,total_checklists,with_birds,observers,without_birds,pct_with_birds,pct_without_birds
54,NORWOOD PARK,829,793,170,36,95.657419,4.342581
49,NEAR WEST SIDE,1276,1223,351,53,95.846395,4.153605
31,HERMOSA,335,322,67,13,96.119403,3.880597
17,DOUGLAS,375,365,62,10,97.333333,2.666667
29,GREATER GRAND CROSSING,120,117,60,3,97.500000,2.500000
18,DUNNING,798,780,120,18,97.744361,2.255639
75,WEST TOWN,7093,6936,956,157,97.786550,2.213450
8,BELMONT CRAGIN,1431,1403,176,28,98.043326,1.956674
7,AVONDALE,1851,1815,275,36,98.055105,1.944895
56,OHARE,2133,2093,1099,40,98.124707,1.875293


In [ ]:
# compare checklist activity across observer accounts
observer_rank = (
    checklists.groupby("OBSERVER ID")
    .agg(
        total_checklists=("SAMPLING EVENT IDENTIFIER", "nunique"),
        with_birds=("has_bird_record", "sum")
    )
    .reset_index()
)

observer_rank["without_birds"] = (
    observer_rank["total_checklists"] - observer_rank["with_birds"]
)

observer_rank["pct_with_birds"] = (
    100 * observer_rank["with_birds"] / observer_rank["total_checklists"]
)

print("Most active accounts:")
display(observer_rank.nlargest(15, "total_checklists"))

print("Accounts with the most unmatched checklists:")
display(observer_rank.nlargest(15, "without_birds"))

Most active accounts:


,OBSERVER ID,total_checklists,with_birds,without_birds,pct_with_birds
7748,obsr4476574,12405,12197,208,98.323257
3103,obsr211276,7058,6937,121,98.285633
6114,obsr3367958,3066,2989,77,97.488584
881,obsr1321437,2727,2709,18,99.339934
8827,obsr562473,1900,1898,2,99.894737
172,obsr106714,1852,1850,2,99.892009
5488,obsr3164036,1785,1772,13,99.271709
3577,obsr239859,1630,1629,1,99.938650
10088,obsr923415,1625,1584,41,97.476923
1181,obsr142929,1591,1588,3,99.811439


Accounts with the most unmatched checklists:


,OBSERVER ID,total_checklists,with_birds,without_birds,pct_with_birds
5768,obsr3222566,609,399,210,65.517241
7748,obsr4476574,12405,12197,208,98.323257
3103,obsr211276,7058,6937,121,98.285633
5981,obsr3289262,918,829,89,90.305011
6114,obsr3367958,3066,2989,77,97.488584
10088,obsr923415,1625,1584,41,97.476923
881,obsr1321437,2727,2709,18,99.339934
8638,obsr5368277,1267,1249,18,98.579321
2517,obsr1846214,124,108,16,87.096774
1193,obsr1432780,24,9,15,37.500000


In [ ]:
# measure how much of each area's reporting comes from this observer
top_observer = (
    checklists[
        checklists["OBSERVER ID"] == "obsr4476574"
    ]
    .groupby("community")
    .size()
    .reset_index(name="top_observer_checklists")
)

community_rank = community_rank.merge(
    top_observer,
    on="community",
    how="left"
)

community_rank["top_observer_checklists"] = (
    community_rank["top_observer_checklists"].fillna(0).astype(int)
)

community_rank["pct_top_observer"] = (
    100 * community_rank["top_observer_checklists"]
    / community_rank["total_checklists"]
)

display(
    community_rank.nlargest(15, "pct_top_observer")[
        [
            "community",
            "total_checklists",
            "top_observer_checklists",
            "pct_top_observer"
        ]
    ]
)

,community,total_checklists,top_observer_checklists,pct_top_observer
21,EDGEWATER,9282,6597,71.073045
60,ROGERS PARK,6634,2982,44.950256
74,WEST RIDGE,4218,1345,31.887150
66,UPTOWN,39441,1464,3.711873
39,LINCOLN SQUARE,9929,17,0.171216
0,ALBANY PARK,5721,0,0.000000
1,ARCHER HEIGHTS,328,0,0.000000
2,ARMOUR SQUARE,253,0,0.000000
3,ASHBURN,255,0,0.000000
4,AUBURN GRESHAM,20,0,0.000000
